In [1]:
from utility.huggingface import suggest_automodel_class
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from baselines.dnnmem import DNNmem

/home/glaswegian/miniconda3/envs/schedtune/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Failed to import transformers.data.data_collator because of the following error (look up to see its traceback):
numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

# Transformers Model

In [1]:
model_name = "facebook/opt-125m"
model_class = suggest_automodel_class(model_name)
model = model_class.from_pretrained(model_name)

NameError: name 'suggest_automodel_class' is not defined

In [3]:

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# 2. 加载数据集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. 分词
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. 创建 DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. 创建 DataLoader
tokenized_datasets.set_format("torch")
dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True, collate_fn=data_collator)


# CNN Model

In [4]:
from perf_estimator.models import AllModels
from perf_estimator.dataset import image_dataset

In [5]:
cnn_model = AllModels["ResNet50"].value
cnn_dl = image_dataset()

# Estimation

## DNNmem

In [6]:
cnn_est = DNNmem(
    model=cnn_model,
    dataloader=cnn_dl,
    max_est_memory_in_bytes=8*1024*1024*1024,  # 8GB
    optimizer=torch.optim.SGD,
    is_transformer=False
)
cnn_est.estimate()
print(f"Estimated memory: {cnn_est.estimate_memory} bytes, and execute time: {cnn_est.execute_time} ns")


Execute the forward and backward analysis first.
Estimated memory: 3189768192 bytes, and execute time: 28653738265 ns


In [7]:
transformer = DNNmem(
    model=model,
    dataloader=dataloader,
    max_est_memory_in_bytes=8*1024*1024*1024,  # 8GB
    optimizer=torch.optim.AdamW,
    is_transformer=True
)
transformer.estimate()
print(f"Estimated memory: {transformer.estimate_memory} bytes, and execute time: {transformer.execute_time} ns")

Estimated memory: 3206545408 bytes, and execute time: 1157493736 ns


## Schedtune

In [6]:
from baselines.schedtune import ScheduleTune
cnn_sched = ScheduleTune(
    model=cnn_model,
    dataloader=cnn_dl,
    optimizer=torch.optim.SGD,
    is_transformer=False,
    device_id=0,
)
cnn_sched.estimate()
print(f"Estimated memory: {cnn_sched.estimate_memory} bytes, and execute time: {cnn_sched.execute_time} ns")

ModuleNotFoundError: No module named 'sklearn'